# Coconut audit (ledger §127 C)

**No training.** The third-party `UdeSA/coconut-0.2B-gsm8k-aug` checkpoint (GPT-2, six continuous thoughts,
34.57% on GSM8K; no official Coconut GSM8K checkpoint exists). Gate, twins on both profiles, masking and
gold-value repair, all with native-decoding outcomes. Prediction P13 is in the ledger. No dataset to attach.
Publish the output when finished (Output tab -> New Dataset).

## Setup

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "3323329ceebf0e02b7467c6408fdc0483a834c87"
REPO_DIR = "/kaggle/working/latent-reasoning"
OUTPUT_ROOT = "/kaggle/working/coconut_audit"
RUN_SMOKE = True

import glob, json, os, pathlib, subprocess, sys, time
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "300")
assert len(RUN_COMMIT) == 40 and RUN_COMMIT != "0" * 40
if not pathlib.Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--tags"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", RUN_COMMIT], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("code commit", subprocess.run(["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip())
print("gpu", subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())

### Environment and focused tests

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "tpot"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.52.4", "datasets==3.6.0", "peft==0.15.2", "accelerate==1.7.0",
                "huggingface-hub>=0.34,<1.0", "safetensors==0.5.3", "pytest>=8,<10"], check=True)
probe = subprocess.run([sys.executable, "-c",
    "from peft.import_utils import is_torchao_available\ntry:\n print(is_torchao_available())\nexcept ImportError:\n print('incompatible')"],
    capture_output=True, text=True)
if "incompatible" in probe.stdout:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", *["tests/test_coconut.py", "tests/test_latent_repair_runner.py", "tests/test_chain_pools.py", "tests/test_cache_carrier.py"]], check=True)

def run_with_retry(command, attempts=3, wait=60):
    for attempt in range(1, attempts + 1):
        result = subprocess.run(command)
        if result.returncode == 0:
            return
        if attempt == attempts:
            raise subprocess.CalledProcessError(result.returncode, command)
        print(f"runner exited with {result.returncode}; retrying in {wait}s ({attempt + 1}/{attempts})")
        time.sleep(wait)

## The audit (smoke first, then full)

In [ ]:
def audit(out, smoke):
    cmd = [sys.executable, "-u", "scripts/run_coconut_audit.py", "--output-dir", out, "--batch-size", "32", "--precision", "float32", "--device", "cuda"]
    if smoke:
        cmd.append("--smoke")
    run_with_retry(cmd)
    return json.loads(open(pathlib.Path(out) / "summary.json").read())

if RUN_SMOKE:
    audit(OUTPUT_ROOT + "_smoke", True)
summary = audit(OUTPUT_ROOT, False)
print(json.dumps(summary["gate"], indent=2)); print("transport long", summary["transport_long"]); print(json.dumps(summary["checks"], indent=2))
print("repair primary", {k: v["rate"] for k, v in summary["repair"]["primary"].items()}); print("stage seconds", summary["stage_seconds"])
print("Reminder: publish this notebook's output as a dataset (Output tab -> New Dataset).")